# 🏈 NFL Draft Scouting Alpha — NLP Classification Pipeline

> **Goal:** Find the *Hidden Delta* between a scout's numerical grade and their text.
> Identify the *Linguistic Success Centroid* for each position group and classify every
> player into four profile buckets: **Clean Hit**, **Hidden Gem**,
> **Grade-Text Mismatch (Overrated)**, or **Hedge Risk**.

**Training window:** 2014–2021 NFL Drafts
**Data:** `data/processed/draft_enriched_with_contracts.csv`
**Outcome variable:** `made_it_contract` — 1 = earned a 2nd contract, 0 = did not
**Key design decision:** "Overrated" threshold uses the **top-25th percentile grade per position group**
(not a global cutoff), making mismatch detection position-aware.

| Section | Content |
|---------|---------|
| 1 | Data Loading & Preprocessing |
| 2 | Keyword Taxonomy Definition |
| 3 | TF-IDF Vectorization & Cosine Similarity |
| 4 | Top 5 Success / Bust Anchor Words Per Position |
| 5 | Mismatch Profile Detection |
| 6 | Predictive Power Analysis |
| 7 | Output Summary |


## Section 1 — Data Loading & Preprocessing

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import re, warnings
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

OUTDIR = Path('.')   # change to save figures elsewhere
print("✅ Imports OK")


In [ ]:
# ── Load & filter data ────────────────────────────────────────────────────────
ROOT      = Path('..') if Path('..').joinpath('data').exists() else Path('.')
DATA_PATH = ROOT / 'data' / 'processed' / 'draft_enriched_with_contracts.csv'

raw = pd.read_csv(DATA_PATH, low_memory=False)
df  = raw.query('2014 <= year <= 2021').copy()

text_cols = ['overview', 'strengths', 'weaknesses']
df[text_cols] = df[text_cols].fillna('')
df['scouting_text'] = (
    df[text_cols].agg(' '.join, axis=1)
    .str.replace(r'\s+', ' ', regex=True).str.strip()
)
df['grade']            = pd.to_numeric(df['grade'], errors='coerce')
df                     = df[df['grade'].between(4.0, 8.5)]   # remove junk outliers
df['made_it_contract'] = df['made_it_contract'].map({True:1, False:0, 1:1, 0:0})
df = df.dropna(subset=['scouting_text','grade','made_it_contract','Pos_Group'])
df = df[df['scouting_text'].str.strip() != ''].reset_index(drop=True)

TARGET_POSITIONS = ['EDGE','OL','WR','QB','RB','DB','DT','LB','TE']
pos_df = df[df['Pos_Group'].isin(TARGET_POSITIONS)].copy().reset_index(drop=True)

print(f"Players: {len(pos_df)}  |  Draft classes: 2014–2021")
print(f"Hits: {int(pos_df['made_it_contract'].sum())} ({pos_df['made_it_contract'].mean():.1%})  |  "
      f"Busts: {int((1-pos_df['made_it_contract']).sum())} ({(1-pos_df['made_it_contract']).mean():.1%})")
print("\nPosition group breakdown:")
g = pos_df.groupby('Pos_Group').agg(n=('player_name','count'),
    hit_rate=('made_it_contract','mean'),
    grade_med=('grade','median'))
g['hit_rate'] = g['hit_rate'].map('{:.1%}'.format)
g['grade_med'] = g['grade_med'].round(2)
print(g.to_string())


### NFL-Aware Text Preprocessing

Adapted from `legacy_annual_tf_idf.ipynb`. Key decisions:

| Step | Rationale |
|------|-----------|
| **KEEP_WORDS** — un-stop directional adjectives | "high" (pad level), "low" (leverage), "deep" (route depth) carry scouting signal |
| **CUSTOM_STOPS** — remove generic filler | "player", "ability", "prospect" appear in virtually every report; zero discriminative value |
| **PHRASE_BLOCKLIST** — strip outcome-leaking phrases before tokenization | "pro bowl", "practice squad" directly reveal outcome |
| **Hyphen normalization → space** | "hand-fighting" becomes "hand fighting", captured as TF-IDF bigram |
| **Suffix-stripping lemmatization** | Covers plurals, -ing/-ed/-ly/-tion forms without external downloads |


In [ ]:
# ── NFL-Aware Preprocessing (self-contained, no NLTK downloads required) ──────
# Standard English stop words (bundled inline — avoids NLTK network dependency)
_STD_STOPS = {
    'i','me','my','myself','we','our','ours','ourselves','you',"you're","you've",
    "you'll","you'd",'your','yours','yourself','yourselves','he','him','his',
    'himself','she',"she's",'her','hers','herself','it',"it's",'its','itself',
    'they','them','their','theirs','themselves','what','which','who','whom',
    'this','that',"that'll",'these','those','am','is','are','was','were','be',
    'been','being','have','has','had','having','do','does','did','doing','a',
    'an','the','and','but','if','or','because','as','until','while','of','at',
    'by','for','with','about','against','between','into','through','during',
    'before','after','above','below','to','from','in','out','on','off','over',
    'under','again','further','then','once','here','there','when','where','why',
    'how','all','both','each','few','more','most','other','some','such','no',
    'nor','not','only','own','same','so','than','too','very','s','t','can',
    'will','just','don',"don't",'should',"should've",'now','d','ll','m','o',
    're','ve','y','ain',
}

KEEP_WORDS = {
    'high','low','heavy','light','deep','short','long','wide',
    'hard','soft','strong','quick','good','great','up','down',
    'off','out','over','through','above','below',
}

CUSTOM_STOPS = {
    'prospect','player','players','show','shows','need','needs',
    'ability','also','often','must','well','still','use','get',
    'make','look','help','work','time','year','team','game',
    'continue','develop','development','nfl','draft','college',
    'level','type',
}

PHRASE_BLOCKLIST = [
    'undrafted free agent','practice squad','free agent','early starter',
    'pro bowl','late round','undrafted free','make roster','rostered',
]

NFL_STOPWORDS = (_STD_STOPS - KEEP_WORDS) | CUSTOM_STOPS

# Lightweight suffix-stripping lemmatizer
_RULES = [('ies','y'),('es',''),('s',''),('ing',''),('ed',''),
          ('ly',''),('tion',''),('ness',''),('ment','')]

def _lemma(word: str) -> str:
    for suffix, rep in _RULES:
        if word.endswith(suffix) and len(word) - len(suffix) >= 3:
            return word[:-len(suffix)] + rep
    return word

def nfl_preprocess(text: str) -> str:
    \"\"\"NFL-aware preprocessing pipeline.

    Steps: lowercase → strip outcome-leaking phrases → normalize hyphens →
    alpha-only → tokenize → NFL stopwords → suffix-strip lemmatize.
    \"\"\"
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower()
    for phrase in sorted(PHRASE_BLOCKLIST, key=len, reverse=True):
        text = text.replace(phrase, ' ')
    text = re.sub(r'[-\u2013\u2014]', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in NFL_STOPWORDS and len(t) > 2]
    tokens = [_lemma(t) for t in tokens]
    return ' '.join(tokens)

pos_df['clean_text'] = pos_df['scouting_text'].apply(nfl_preprocess)
pos_df = pos_df[pos_df['clean_text'].str.strip() != ''].reset_index(drop=True)

tok = pos_df['clean_text'].str.split().str.len()
print(f"Players after preprocessing: {len(pos_df)}")
print(f"Tokens/player — median: {int(tok.median())}, mean: {tok.mean():.0f}, "
      f"min: {tok.min()}, max: {tok.max()}")
print("\nSample (cleaned, 200 chars):")
print(pos_df['clean_text'].iloc[0][:200])


## Section 2 — Keyword Taxonomy Definition

Four vocabulary clusters capture distinct scouting signal types.

| Cluster | Signal | Interpretation |
|---------|--------|---------------|
| **Raw Athlete** | Physical upside language | Scouts hedging on developmental players — historically high bust signal |
| **Technique** | Football-specific process words | Alpha signal — scouts identifying process-driven contributors |
| **Hedge** | Conditional / uncertain language | Risk flag — scouts softening their conviction |
| **Definitive** | Confident, declarative language | Success signal — scouts committing to current ability |

> **Implementation note:** Terms are designed to match *post-lemmatization* text (base forms).
> Multi-word phrases are counted as substring matches on the full cleaned string.


In [ ]:
# ── Keyword Taxonomy (post-lemmatization compatible) ──────────────────────────
TAXONOMY = {
    'raw_athlete': [
        'trait','upside','frame','basketball','athleticism',
        'raw','tool','developmental','athletic','physical specimen',
    ],
    'technique': [
        'hand fight','leverage','nuance','landmark','stack',
        'footwork','pad level','anchor','technique','assignment',
        'hip','redirect','pursuit','hand placement','leverage point',
    ],
    'hedge': [
        ' but ',' if ','when he','flash','could be','project',
        'once he','need to','inconsistent','struggl','has the potential',
        'will need','must improv',
    ],
    'definitive': [
        'will ','consistent','reliabl','proven',
        'demonstrat','finish','control','dominant','elite',
        'alway','never miss',
    ],
}

def keyword_density(text: str, terms: list) -> float:
    \"\"\"Keyword density = total substring hits / token count.\"\"\"
    if not text or not text.strip():
        return 0.0
    n = len(text.split())
    return sum(text.count(kw) for kw in terms) / n if n else 0.0

for col, terms in TAXONOMY.items():
    pos_df[f'{col}_density'] = pos_df['clean_text'].apply(
        lambda t: keyword_density(t, terms)
    )

density_cols = [f'{c}_density' for c in TAXONOMY]
print("Mean keyword densities by position group:")
dm = pos_df.groupby('Pos_Group')[density_cols].mean().round(4)
dm.columns = [c.replace('_density','') for c in dm.columns]
print(dm.to_string())

# Quick observation
print("\nObservations:")
print(f"  EDGE has highest raw_athlete density ({pos_df[pos_df['Pos_Group']=='EDGE']['raw_athlete_density'].mean():.4f})")
print(f"  OL   has highest technique density  ({pos_df[pos_df['Pos_Group']=='OL']['technique_density'].mean():.4f})")
print(f"  QB   has lowest  raw_athlete density ({pos_df[pos_df['Pos_Group']=='QB']['raw_athlete_density'].mean():.4f})")


## Section 3 — TF-IDF Vectorization & Cosine Similarity

For each position group:
1. Fit a single TF-IDF vectorizer on all reports in that group
2. **Success Centroid** = mean TF-IDF vector of `made_it_contract = 1` players
3. **Bust Centroid** = mean TF-IDF vector of `made_it_contract = 0` players
4. Compute each player's cosine similarity to both centroids

**TF-IDF parameters (from legacy pipeline):**
- `ngram_range=(1, 2)` — captures "pad level", "hand fighting", "pass rush"
- `max_features=1500` — vocabulary cap to suppress noise
- `min_df=3` — drops hapax legomena and typos
- `sublinear_tf=True` — log-scale TF, stabilises short docs (~140 tokens/report)


In [ ]:
# ── TF-IDF vectorization + centroid computation ───────────────────────────────
TFIDF_PARAMS = dict(max_features=1500, ngram_range=(1,2), min_df=3, sublinear_tf=True)

pos_results = {}
for pos in TARGET_POSITIONS:
    sub = pos_df[pos_df['Pos_Group'] == pos].copy().reset_index(drop=True)
    if len(sub) < 20:
        print(f"⚠️  {pos}: only {len(sub)} players — skipping")
        continue
    vec   = TfidfVectorizer(**TFIDF_PARAMS)
    X     = vec.fit_transform(sub['clean_text'])
    vocab = np.array(vec.get_feature_names_out())
    hit_mask  = sub['made_it_contract'].values == 1
    bust_mask = ~hit_mask
    hit_c  = X[hit_mask].mean(axis=0).A1  if hit_mask.sum()  > 0 else np.zeros(X.shape[1])
    bust_c = X[bust_mask].mean(axis=0).A1 if bust_mask.sum() > 0 else np.zeros(X.shape[1])
    sim_hit  = cosine_similarity(X, hit_c.reshape(1,-1)).ravel()
    sim_bust = cosine_similarity(X, bust_c.reshape(1,-1)).ravel()
    pos_results[pos] = dict(sub=sub, vectorizer=vec, X=X, vocab=vocab,
                            hit_centroid=hit_c, bust_centroid=bust_c,
                            sim_hit=sim_hit, sim_bust=sim_bust,
                            n_hits=int(hit_mask.sum()), n_busts=int(bust_mask.sum()))
    print(f"✅ {pos:4s}: {len(sub):4d} players | {hit_mask.sum():3d} hits | "
          f"{bust_mask.sum():3d} busts | vocab={len(vocab)}")

# Attach back to pos_df
pos_df['sim_hit']  = np.nan
pos_df['sim_bust'] = np.nan
for pos, res in pos_results.items():
    mask = pos_df['Pos_Group'] == pos
    pos_df.loc[mask, 'sim_hit']  = res['sim_hit']
    pos_df.loc[mask, 'sim_bust'] = res['sim_bust']

print(f"\nCosine similarity summary:")
print(pos_df.groupby('Pos_Group')[['sim_hit','sim_bust']].mean().round(4).to_string())


In [ ]:
# ── Figure 1: Centroid similarity distributions ───────────────────────────────
npos  = len(pos_results)
ncols = 3; nrows = (npos + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4*nrows))
axes = axes.ravel()

for i, pos in enumerate(pos_results.keys()):
    ax  = axes[i]
    res = pos_results[pos]
    sub = res['sub']
    h   = res['sim_hit'][sub['made_it_contract'].values == 1]
    b   = res['sim_hit'][sub['made_it_contract'].values == 0]
    ax.hist(b, bins=25, alpha=0.55, color='tomato',    label='Bust', density=True)
    ax.hist(h, bins=25, alpha=0.55, color='steelblue', label='Hit',  density=True)
    ax.axvline(0.40, color='gold', linestyle='--', lw=1.5, label='0.40 threshold')
    ax.set_title(f'{pos}  (n={len(sub)}, hits={res["n_hits"]})', fontweight='bold', fontsize=10)
    ax.set_xlabel('Cosine Sim → Success Centroid', fontsize=8)
    ax.legend(fontsize=7)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Cosine Similarity to Success Centroid by Position Group', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTDIR / 'fig1_centroid_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


## Section 4 — Top 5 Success / Bust Anchor Words Per Position

**Method: Log-Odds Ratio**

$$\text{log-odds}(w) = \log\!\left(\frac{\bar{\text{TF-IDF}}(w \mid \text{hit}) + \varepsilon}{\bar{\text{TF-IDF}}(w \mid \text{bust}) + \varepsilon}\right)$$

- ε = 1e-9 avoids log(0)
- Mean TF-IDF per group serves as term probability proxy
- **Positive log-odds** → term appears disproportionately in *hit* reports
- **Negative log-odds** → term appears disproportionately in *bust* reports


In [ ]:
# ── Log-odds anchor words per position ────────────────────────────────────────
EPSILON = 1e-9
TOP_N   = 5
anchor_rows = []

for pos in TARGET_POSITIONS:
    if pos not in pos_results:
        continue
    res      = pos_results[pos]
    sub      = res['sub']
    X        = res['X']
    vocab    = res['vocab']
    hit_mask = sub['made_it_contract'].values == 1
    hit_mean  = X[hit_mask].mean(axis=0).A1  if hit_mask.sum()  > 0 else np.zeros(X.shape[1])
    bust_mean = X[~hit_mask].mean(axis=0).A1 if (~hit_mask).sum() > 0 else np.zeros(X.shape[1])
    log_odds  = np.log((hit_mean + EPSILON) / (bust_mean + EPSILON))
    top_s = np.argsort(log_odds)[::-1][:TOP_N]
    top_b = np.argsort(log_odds)[:TOP_N]
    for rank in range(TOP_N):
        anchor_rows.append(dict(
            pos_group=pos, rank=rank+1,
            success_word=vocab[top_s[rank]], success_log_odds=round(float(log_odds[top_s[rank]]),4),
            bust_word=vocab[top_b[rank]],    bust_log_odds=round(float(log_odds[top_b[rank]]),4),
        ))

anchor_df = pd.DataFrame(anchor_rows)

# Clean summary table
summary = (anchor_df.groupby('pos_group').apply(
    lambda g: pd.Series({
        'Top 5 Success Words': ' | '.join(g.sort_values('rank')['success_word']),
        'Top 5 Bust Words':    ' | '.join(g.sort_values('rank')['bust_word']),
    })).reset_index().rename(columns={'pos_group':'Position'}))
print("Anchor Word Summary:")
for _, row in summary.iterrows():
    print(f"\n  {row['Position']}")
    print(f"    ✅ {row['Top 5 Success Words']}")
    print(f"    ❌ {row['Top 5 Bust Words']}")


In [ ]:
# ── Figure 2: Anchor word heatmaps ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(20, 7))

for ax, word_type, lo_col, cmap in [
    (axes[0], 'success', 'success_log_odds', 'YlGn'),
    (axes[1], 'bust',    'bust_log_odds',    'OrRd'),
]:
    heat = anchor_df.pivot(
        index=f'{word_type}_word', columns='pos_group', values=lo_col
    ).fillna(0)
    heat = heat.loc[heat.abs().mean(axis=1).sort_values(ascending=False).index]
    sns.heatmap(heat, ax=ax, cmap=cmap, annot=True, fmt='.2f',
                linewidths=0.5, cbar_kws={'label':'Log-Odds'})
    ax.set_title(f'Top 5 {"Success" if word_type=="success" else "Bust"} Anchor Words\n'
                 f'(Log-Odds vs. opposite class)',
                 fontweight='bold', fontsize=12)
    ax.set_xlabel('Position Group'); ax.set_ylabel('Term')
    ax.tick_params(axis='both', rotation=0)

plt.tight_layout()
plt.savefig(OUTDIR / 'fig2_anchor_word_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


## Section 5 — Mismatch Profile Detection

**Position-aware grade thresholds** — using each position's own 75th-percentile grade
as the "high grade" cutoff (rather than a global 6.5) ensures fair comparison across
positions with different grade distributions.

| Profile | Condition | Actual hit rate (live run) |
|---------|-----------|---------------------------|
| **Clean Hit** | sim_hit ≥ pos P75, grade ≥ pos P75, low hedge | **77.8%** |
| **Hedge Risk** | grade ≥ pos P75, hedge > pos median | **70.2%** |
| **Grade-Text Mismatch (Overrated)** | grade ≥ pos P75, sim_hit < 0.40 | **38.0%** |
| **Unclassified** | none of the above | 22.3% |
| **Hidden Gem** | grade ≤ pos median, sim_hit > 0.45 | n=1 (threshold sensitivity noted below) |

> **Note on Hidden Gems:** Very few players have *both* a below-median grade *and* high
> cosine similarity to the success centroid (0.45+). The sim_hit threshold may be relaxed
> to 0.40 to surface more candidates — see commented alternative below.


In [ ]:
# ── Per-position thresholds ───────────────────────────────────────────────────
pos_grade_p75 = pos_df.groupby('Pos_Group')['grade'].quantile(0.75).rename('grade_p75')
pos_grade_med = pos_df.groupby('Pos_Group')['grade'].quantile(0.50).rename('grade_med')
pos_hedge_med = pos_df.groupby('Pos_Group')['hedge_density'].median().rename('hedge_median')
pos_sim_p75   = pos_df.groupby('Pos_Group')['sim_hit'].quantile(0.75).rename('sim_p75')

pos_df = (pos_df
          .merge(pos_grade_p75, on='Pos_Group', how='left')
          .merge(pos_grade_med, on='Pos_Group', how='left')
          .merge(pos_hedge_med, on='Pos_Group', how='left')
          .merge(pos_sim_p75,   on='Pos_Group', how='left'))

print("Per-position grade thresholds:")
thresh = pos_df.groupby('Pos_Group')[['grade_med','grade_p75']].first()
thresh.columns = ['Grade Median (Hidden Gem cutoff)', 'Grade P75 (Overrated / Clean Hit cutoff)']
print(thresh.round(2).to_string())


In [ ]:
# ── Assign mismatch profiles ──────────────────────────────────────────────────
def assign_profile(row):
    grade     = row['grade']
    sim       = row['sim_hit']
    hedge_d   = row['hedge_density']
    hedge_med = row['hedge_median']
    g_p75     = row['grade_p75']
    g_med     = row['grade_med']
    s_p75     = row['sim_p75']

    # Priority: Overrated > Hidden Gem > Hedge Risk > Clean Hit > Unclassified
    if grade >= g_p75 and (pd.isna(sim) or sim < 0.40):
        return 'Grade-Text Mismatch (Overrated)'
    if grade <= g_med and not pd.isna(sim) and sim > 0.45:
        # Relax to 0.40 to surface more: sim > 0.40
        return 'Hidden Gem'
    if grade >= g_p75 and hedge_d > hedge_med:
        return 'Hedge Risk'
    if not pd.isna(sim) and sim >= s_p75 and grade >= g_p75 and hedge_d <= hedge_med:
        return 'Clean Hit'
    return 'Unclassified'

pos_df['profile'] = pos_df.apply(assign_profile, axis=1)

print("Profile distribution:")
for prof, grp in pos_df.groupby('profile'):
    print(f"  {prof:35s}: {len(grp):4d}  (hit rate {grp['made_it_contract'].mean():.1%})")


In [ ]:
# ── Figure 3: Profile hit rates ───────────────────────────────────────────────
prof_hr = (pos_df.groupby('profile')['made_it_contract']
           .agg(['mean','count']).reset_index()
           .rename(columns={'profile':'Profile','mean':'hit_rate','count':'n'})
           .sort_values('hit_rate', ascending=True))

cmap_prof = {
    'Clean Hit': 'steelblue', 'Hidden Gem': 'seagreen',
    'Hedge Risk': 'goldenrod',
    'Grade-Text Mismatch (Overrated)': 'tomato',
    'Unclassified': '#aaaaaa',
}
fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(prof_hr['Profile'], prof_hr['hit_rate']*100,
               color=[cmap_prof.get(p,'#aaaaaa') for p in prof_hr['Profile']],
               edgecolor='white', height=0.55)
for bar, (_, row) in zip(bars, prof_hr.iterrows()):
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
            f"{row['hit_rate']*100:.1f}%  (n={int(row['n'])})",
            va='center', fontsize=9)
base = pos_df['made_it_contract'].mean()*100
ax.axvline(base, color='black', linestyle='--', lw=1.2, label=f'Base rate {base:.1f}%')
ax.set_xlabel('2nd Contract Rate (%)')
ax.set_title('Mismatch Profile → Actual Hit Rate (2014–2021)\nPosition-Aware Grade Thresholds',
             fontweight='bold')
ax.set_xlim(0, 90); ax.xaxis.set_major_formatter(mtick.PercentFormatter()); ax.legend()
plt.tight_layout()
plt.savefig(OUTDIR / 'fig3_profile_hit_rates.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Grade distribution with P75 threshold overlay ────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.ravel()
for i, pos in enumerate(TARGET_POSITIONS):
    sub = pos_df[pos_df['Pos_Group'] == pos]
    if len(sub) == 0: axes[i].set_visible(False); continue
    ax = axes[i]
    ax.hist(sub[sub['made_it_contract']==0]['grade'], bins=20, alpha=0.55, color='tomato',    density=True, label='Bust')
    ax.hist(sub[sub['made_it_contract']==1]['grade'], bins=20, alpha=0.55, color='steelblue', density=True, label='Hit')
    p75 = sub['grade_p75'].iloc[0]
    ax.axvline(p75, color='black', linestyle='--', lw=1.5, label=f'P75={p75:.2f}')
    ax.set_title(f'{pos}  (n={len(sub)})', fontweight='bold', fontsize=9)
    ax.set_xlabel('Grade', fontsize=8); ax.legend(fontsize=7)
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Grade Distributions by Position (dashed = P75 "Overrated" threshold)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTDIR / 'fig6_grade_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


## Section 6 — Predictive Power Analysis

**Two logistic regression models compared via 5-fold stratified cross-validated ROC-AUC:**

| Model | Features |
|-------|---------|
| **Baseline (Grade Only)** | `grade` |
| **Scouting Alpha** | `grade` + `sim_hit` + `raw_athlete_density` + `technique_density` + `hedge_density` + `definitive_density` |

Live results: **AUC 0.7160 → 0.7428 (+0.027)** — a meaningful lift given that grade already
carries most of the signal. Strongest NLP addition is `sim_hit` (+0.028 AUC on its own),
indicating that text-to-centroid alignment is the single best complement to grade.


In [ ]:
# ── Build modelling dataset ───────────────────────────────────────────────────
model_df = pos_df.dropna(subset=['sim_hit','grade','made_it_contract']).copy()
model_df['made_it_contract'] = model_df['made_it_contract'].astype(int)

FEATURE_COLS = ['grade','sim_hit','raw_athlete_density','technique_density',
                'hedge_density','definitive_density']
GRADE_ONLY   = ['grade']

X_full  = model_df[FEATURE_COLS].values
X_grade = model_df[GRADE_ONLY].values
y       = model_df['made_it_contract'].values

print(f"Model set: {len(model_df)} players | "
      f"{y.sum()} hits ({y.mean():.1%}) | {(1-y).sum()} busts")


In [ ]:
# ── Cross-validated AUC comparison ───────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def cv_auc(X, y, cv):
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(max_iter=2000, solver='lbfgs',
                                  class_weight='balanced', random_state=42)),
    ])
    return cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc')

auc_full  = cv_auc(X_full,  y, cv)
auc_grade = cv_auc(X_grade, y, cv)

print("=" * 55)
print(f"  Grade-Only AUC:           {auc_grade.mean():.4f} ± {auc_grade.std():.4f}")
print(f"  Scouting Alpha AUC:       {auc_full.mean():.4f} ± {auc_full.std():.4f}")
print(f"  NLP Delta:                {auc_full.mean()-auc_grade.mean():+.4f}")
print("=" * 55)
print(f"\nFold AUC — Grade:  {[f'{s:.3f}' for s in auc_grade]}")
print(f"Fold AUC — Alpha:  {[f'{s:.3f}' for s in auc_full]}")


In [ ]:
# ── Incremental lift: add one NLP feature at a time ──────────────────────────
incremental = []
for feat in FEATURE_COLS[1:]:
    auc_inc = cv_auc(model_df[['grade', feat]].values, y, cv)
    incremental.append({'NLP Feature': feat,
                        'AUC (grade + feature)': round(float(auc_inc.mean()),4),
                        'Lift vs grade-only':    round(float(auc_inc.mean()-auc_grade.mean()),4)})
inc_df = pd.DataFrame(incremental).sort_values('Lift vs grade-only', ascending=False)
print("Incremental AUC lift (grade + one NLP feature):")
print(inc_df.to_string(index=False))
print(f"\n→ Best single NLP addition: '{inc_df.iloc[0]['NLP Feature']}' "
      f"(Δ AUC = {inc_df.iloc[0]['Lift vs grade-only']:+.4f})")


In [ ]:
# ── Feature coefficients (standardized logistic regression) ──────────────────
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_full)
lr_full  = LogisticRegression(max_iter=2000, solver='lbfgs',
                               class_weight='balanced', random_state=42)
lr_full.fit(X_scaled, y)
coef_df = pd.DataFrame({'Feature': FEATURE_COLS, 'Coefficient': lr_full.coef_[0]})            .sort_values('Coefficient', ascending=False)
print("Logistic regression coefficients (standardized; positive → more hits):")
print(coef_df.to_string(index=False))

# Fig 4: coefficients
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['steelblue' if c > 0 else 'tomato' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
for i, (_, r) in enumerate(coef_df.iterrows()):
    ax.text(r['Coefficient'] + (0.01 if r['Coefficient'] >= 0 else -0.01),
            i, f"{r['Coefficient']:+.3f}", va='center',
            ha='left' if r['Coefficient'] >= 0 else 'right', fontsize=8)
ax.set_xlabel('Standardized Coefficient  (positive → more hits)')
ax.set_title('Scouting Alpha Feature Importance\n(Logistic Regression, balanced classes)',
             fontweight='bold')
plt.tight_layout()
plt.savefig(OUTDIR / 'fig4_feature_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Per-position AUC breakdown ────────────────────────────────────────────────
pos_auc_rows = []
for pos in TARGET_POSITIONS:
    sub = model_df[model_df['Pos_Group'] == pos]
    if len(sub) < 30 or sub['made_it_contract'].nunique() < 2:
        continue
    y_pos  = sub['made_it_contract'].values
    folds  = min(5, max(2, int(y_pos.sum() // 5)))
    cv_pos = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)
    try:
        auc_g = cv_auc(sub[GRADE_ONLY].values,   y_pos, cv_pos)
        auc_a = cv_auc(sub[FEATURE_COLS].values, y_pos, cv_pos)
        pos_auc_rows.append(dict(pos=pos, grade_auc=round(float(auc_g.mean()),4),
                                 alpha_auc=round(float(auc_a.mean()),4),
                                 delta=round(float(auc_a.mean()-auc_g.mean()),4), n=len(sub)))
    except Exception as e:
        print(f"⚠️  {pos}: {e}")
pos_auc_df = pd.DataFrame(pos_auc_rows)

print("Per-position AUC:")
print(pos_auc_df.to_string(index=False))

# Fig 5: per-position AUC
if not pos_auc_df.empty:
    fig, ax = plt.subplots(figsize=(11, 4.5))
    x = np.arange(len(pos_auc_df)); w = 0.35
    ax.bar(x-w/2, pos_auc_df['grade_auc'], w, label='Grade Only',       color='lightcoral', zorder=2)
    ax.bar(x+w/2, pos_auc_df['alpha_auc'], w, label='Grade + NLP Alpha', color='steelblue',  zorder=2)
    for xi, row in zip(x, pos_auc_df.itertuples()):
        ax.annotate(f'{row.delta:+.3f}', xy=(xi+w/2, row.alpha_auc+0.006),
                    ha='center', va='bottom', fontsize=8,
                    color='green' if row.delta > 0 else 'firebrick', fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(pos_auc_df['pos'], fontsize=10)
    ax.set_ylabel('ROC-AUC'); ax.set_ylim(0.35, 0.95)
    ax.axhline(0.5, color='gray', linestyle='--', lw=0.9, label='Random baseline')
    ax.legend()
    ax.set_title('ROC-AUC: Grade-Only vs. Scouting Alpha (5-fold CV)', fontweight='bold')
    ax.grid(axis='y', alpha=0.4)
    plt.tight_layout()
    plt.savefig(OUTDIR / 'fig5_auc_by_position.png', dpi=150, bbox_inches='tight')
    plt.show()


## Section 7 — Output Summary

In [ ]:
# ── Position-by-position summary ─────────────────────────────────────────────
print("=" * 72)
print("SCOUTING ALPHA — POSITION GROUP SUMMARY (2014–2021 NFL Drafts)")
print("=" * 72)

for pos in TARGET_POSITIONS:
    if pos not in pos_results:
        continue
    sub = pos_df[pos_df['Pos_Group'] == pos]
    pa  = anchor_df[anchor_df['pos_group'] == pos].sort_values('rank')
    pa_row = pos_auc_df[pos_auc_df['pos'] == pos] if not pos_auc_df.empty else pd.DataFrame()
    auc_str = (f"Grade AUC={pa_row['grade_auc'].values[0]:.3f} | "
               f"Alpha AUC={pa_row['alpha_auc'].values[0]:.3f} | "
               f"Δ={pa_row['delta'].values[0]:+.3f}") if len(pa_row) else "N/A"

    gems = (sub[sub['profile']=='Hidden Gem']
            .sort_values('sim_hit', ascending=False)
            [['player_name','year','grade','sim_hit','technique_density','made_it_contract']]
            .head(5))
    over = (sub[sub['profile']=='Grade-Text Mismatch (Overrated)']
            .sort_values(['grade','sim_hit'], ascending=[False,True])
            [['player_name','year','grade','sim_hit','raw_athlete_density','made_it_contract']]
            .head(5))

    print(f"\n{'─'*70}")
    print(f"  {pos}  |  n={len(sub)}  |  hit rate={sub['made_it_contract'].mean():.1%}  |  {auc_str}")
    print(f"  Grade P25/P50/P75: {sub['grade'].quantile(0.25):.2f} / "
          f"{sub['grade'].quantile(0.50):.2f} / {sub['grade'].quantile(0.75):.2f}")
    print(f"\n  ✅ Success anchor words:  {' | '.join(pa['success_word'].tolist())}")
    print(f"  ❌ Bust anchor words:     {' | '.join(pa['bust_word'].tolist())}")

    if not gems.empty:
        print(f"\n  💎 Hidden Gems:")
        for _, r in gems.iterrows():
            outcome = '✅' if r['made_it_contract']==1 else '❌'
            print(f"      {r['player_name']:28s}  {int(r['year'])}  grade={r['grade']:.2f}  "
                  f"sim={r['sim_hit']:.3f}  {outcome}")
    else:
        print(f"\n  💎 Hidden Gems: none at current threshold (try relaxing sim_hit to 0.40)")

    if not over.empty:
        print(f"\n  ⚠️  Grade-Text Mismatch (Overrated):")
        for _, r in over.iterrows():
            outcome = '✅' if r['made_it_contract']==1 else '❌'
            print(f"      {r['player_name']:28s}  {int(r['year'])}  grade={r['grade']:.2f}  "
                  f"sim={r['sim_hit']:.3f}  {outcome}")
print(f"\n{'='*72}")


In [ ]:
# ── Export all outputs ────────────────────────────────────────────────────────
anchor_df.to_csv(OUTDIR / 'scouting_alpha_anchor_words.csv', index=False)
print("✅ scouting_alpha_anchor_words.csv")

out_cols = ['year','player_name','Pos_Group','grade','sim_hit','sim_bust',
            'raw_athlete_density','technique_density','hedge_density',
            'definitive_density','profile','made_it_contract']
pos_df[out_cols].to_csv(OUTDIR / 'scouting_alpha_player_profiles.csv', index=False)
print("✅ scouting_alpha_player_profiles.csv")

if not pos_auc_df.empty:
    pos_auc_df.to_csv(OUTDIR / 'scouting_alpha_pos_auc.csv', index=False)
    print("✅ scouting_alpha_pos_auc.csv")

inc_df.to_csv(OUTDIR / 'scouting_alpha_incremental_auc.csv', index=False)
print("✅ scouting_alpha_incremental_auc.csv")


---
## Schema Reference

| Column | Type | Description |
|--------|------|-------------|
| `year` | int | Draft year (2014–2021 for training) |
| `player_name` | str | Player full name |
| `Pos_Group` | str | EDGE, OL, WR, QB, RB, DB, DT, LB, TE |
| `grade` | float | Scout numerical grade (typical range 5.0–7.5) |
| `overview` | str | Report overview section |
| `strengths` | str | Report strengths section |
| `weaknesses` | str | Report weaknesses section |
| `made_it_contract` | int | 1 = earned 2nd contract, 0 = did not |

Update `DATA_PATH` in Section 1 to point at a new data file. The `TARGET_POSITIONS`
list and `TAXONOMY` keyword lists can be extended without changing downstream logic.

## Interpretation Notes

- **sim_hit** is the most powerful single NLP signal (+0.027 AUC on its own).
  It captures holistic text-to-outcome alignment better than any single keyword cluster.
- **`definitive_density` has a *negative* coefficient** in the logistic regression —
  possibly because scouts over-use definitive language on high-profile prospects
  who then underperform expectations. Worth investigating further.
- **Hidden Gem count is very low** (n=1) at the 0.45 sim_hit threshold. Relaxing
  to 0.40 surfaces ~30–40 candidates. The scarcity itself is informative: most
  under-graded players also have weaker text alignment.
- The **TE position shows the largest AUC gain** (+0.089), suggesting scout language
  is especially predictive for tight ends beyond the grade alone.
